# Pipeline
0. Get the list of required swc files
1. Load labels parquet
2. Import the swc file
3. simplify swc file
4. attach synapse labels + neuron type
5. save simplified file
6. convert to json for find-clumpiness
7. save json file
8. calculate clumpiness for each internal node
9. attach results to the labeled swc file
10. save results.

---
# Preprocessing step
1. Create metadata labels for each swc file
2. Look for the releveant swc files only (with the wanted type)
3. Unify them via the already created function in feather file ->>> Improvement

In [1]:
import os
import json
import numpy as np
import pandas as pd
import polars as pl
from tqdm import tqdm
from scripts.helpers import mkdir
from joblib import Parallel, delayed
from scripts.preprocessing import simplify_swc_topology, swc2json, get_neurons_info


# Type data located in the 

path_swc_labels = os.path.join("data", "input_labels", "neuron_data_full_article_princeton.ftr")
swc_labels = pd.read_feather(path_swc_labels)

required_labels = ["super_class", ["central", "optic", "visual_centrifugal", "visual_projection"]]
swc_labels = swc_labels.loc[swc_labels[required_labels[0]].isin(required_labels[1])]

-----
# Single-file hard coded pipeline example

In [ ]:
nueron_itr = 720575940609102805


####################################################################################################
#  1. Load labels parquet
# Parquet labels path
prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

# Load exactly the labels of the example swc file
parquet_labels = pl.scan_parquet(prquet_labels_path)

# Only the relevnt column in the parquet file
labels_parquet = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(nueron_itr)).collect().to_pandas()


####################################################################################################
#  2. Import the swc file
neuron_path = os.path.join("data","input_swc", "sk_lod1_783_healed", f"{nueron_itr}.swc")
neuron_swc = pd.read_csv(neuron_path, 
                         comment='#', 
                         header=None, 
                         sep=r'\s+', 
                         names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


####################################################################################################
#  3. simplify swc file
simple_swc = simplify_swc_topology(neuron_swc, swc_name=f"{nueron_itr}", save_csv=False)


####################################################################################################
#  4. attach synapse labels + neuron type
swc_labeled = pd.merge(left=simple_swc, 
                       right=labels_parquet[["node_id", "type"]].drop_duplicates(), 
                       left_on="node_id", 
                       right_on="node_id", 
                       how="left")


####################################################################################################
#  5. save simplified file
for i in ["data", os.path.join("data", "input_swc"), os.path.join("data", "input_swc", "simplified")]:
    if os.path.exists(i) is False:
        os.mkdir(i)

save_path = os.path.join("data", "input_swc", "simplified", f"{nueron_itr}.csv")
swc_labeled.to_csv(save_path)


####################################################################################################
#  6. convert to json for find-clumpiness
swc2json(swc_dataset=swc_labeled,
         neuron_id=nueron_itr,
         save_json=True,
         save_path=os.path.join("data", "output_json"))


----
# Automated script pipeline


In [ ]:
def process_neuron(neuron_itr):
    # Force Polars to use a single thread to prevent nested parallelism crashes
    os.environ["POLARS_MAX_THREADS"] = "4"

    try:
        #########################
        #  1. Load labels parquet
        # Parquet labels path
        prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

        # Load exactly the labels of the example swc file
        parquet_labels = pl.scan_parquet(prquet_labels_path)

        # Only the relevnt column in the parquet file
        labels_parquet = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(neuron_itr)).collect().to_pandas()

        #########################
        #  2. Import the swc file
        neuron_path = os.path.join("data","input_swc", "sk_lod1_783_healed", f"{neuron_itr}.swc")
        neuron_swc = pd.read_csv(neuron_path, 
                                 comment='#', 
                                 header=None, 
                                 sep=r'\s+', 
                                 names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


        #######################
        #  3. simplify swc file
        simple_swc = simplify_swc_topology(neuron_swc, swc_name=f"{neuron_itr}", save_csv=False)  


        #########################################
        #  4. attach synapse labels + neuron type
        swc_labeled = pd.merge(left=simple_swc, 
                               right=labels_parquet[["node_id", "type"]].drop_duplicates(), 
                               left_on="node_id", 
                               right_on="node_id", 
                               how="left")   

        ##########################
        #  5. save simplified file
        for i in ["data", os.path.join("data", "input_swc"), os.path.join("data", "input_swc", "simplified")]:
            if os.path.exists(i) is False:
                os.mkdir(i)

        save_path = os.path.join("data", "input_swc", "simplified", f"{neuron_itr}.csv")
        swc_labeled.to_csv(save_path)

        return f"Success: {neuron_itr}"

    except Exception as e:
        return f"Error on {neuron_itr}: {e}"


if __name__ == '__main__':
    # Define paths
    swc_path = os.path.join("data", "input_swc", "sk_lod1_783_healed")
    labels_path = os.path.join("data", "input_labels", "processed_swc_data_princeton")
    prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

    # Safe folder creation before multiprocessing starts to avoid race conditions
    folders_to_create = ["data", 
                         os.path.join("data", "input_swc"), 
                         os.path.join("data", "input_swc", "simplified"),
                         os.path.join("data", "output_json")]
    
    for folder in folders_to_create:
        os.makedirs(folder, exist_ok=True)

    # Getting a list of the aviable SWC file in the swc input folder
    swc_files = [i.split(".")[0] for i in os.listdir(swc_path)]

    # Creating parquete file -> only relevent swc file by super-type
    get_neurons_info(overwrite_parquet=False)

    # Getting relevent swc
    parquet_labels = pl.scan_parquet(prquet_labels_path)
    labels_parquet = parquet_labels.select(["neuron"]).collect().to_pandas()

    # Getting list of relevent + real SWC file
    swc_relv = np.intersect1d(swc_files, labels_parquet.neuron.values)
    
    # Select the batch you want to run
    tasks = swc_relv[:20] 
    
    print(f"Starting processing of {len(tasks)} neurons...")
    
    # Execute in parallel using Joblib
    # n_jobs=4 limits the pool to 4 cores to prevent memory exhaustion. 
    # You can increase this if your system has plenty of RAM.
    results = Parallel(n_jobs=4, backend="loky")(delayed(process_neuron)(neuron) for neuron in tqdm(tasks))
    
    # Print any errors that were caught during execution
    for res in results:
        if "Error" in res:
            print(res)


> Function execution halted, old `swc_labels.parquet` file preserved.
Starting processing of 100 neurons...


 30%|███       | 30/100 [00:15<00:40,  1.74it/s]c:\Users\Daniel\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
100%|██████████| 100/100 [01:02<00:00,  1.60it/s]


In [14]:
import pandas as pd
import numpy as np
from itertools import combinations_with_replacement

def calculate_exact_node_clumpiness(swc_df):
    """
    Calculates node-level clumpiness using the exact formulation from 
    Schwartz et al. (2016). Restricts scores exclusively to internal nodes 
    and the root node (parent == -1), returning NaN for all leaf nodes.
    """
    df = swc_df.copy()
    
    # 1. Parse multi-labels (handles "pre post" or lists)
    df['type'] = df['type'].astype(str).apply(
        lambda x: [lbl.strip() for lbl in x.replace(',', ' ').split() if lbl.strip()] 
        if pd.notna(x) and x != 'nan' else []
    )
    
    all_labels = sorted(list(set(lbl for sublist in df['type'] for lbl in sublist)))
    label_combos = list(combinations_with_replacement(all_labels, 2))
    
    # 2. Build Topology Maps
    children_map = {}
    parent_map = {}
    num_children = {}
    is_leaf = {node: True for node in df['node_id']}
    
    for _, row in df.iterrows():
        node = row['node_id']
        parent = row['parent']
        parent_map[node] = parent
        if pd.notna(parent) and parent != -1:
            if parent not in children_map:
                children_map[parent] = []
            children_map[parent].append(node)
            is_leaf[parent] = False
            
    for node, children in children_map.items():
        num_children[node] = len(children)

    roots = df[df['parent'] == -1]['node_id'].tolist()
    
    # Identify all valid relevant leaves M (excluding leaves directly attached to root)
    root_children = set()
    for r in roots:
        root_children.update(children_map.get(r, []))
    
    relevant_leaves = [row['node_id'] for _, row in df.iterrows() 
                       if is_leaf[row['node_id']] and row['parent'] not in roots and len(row['type']) > 0]

    # 3. Descendant Leaf Mapping
    descendant_leaves = {node: set() for node in df['node_id']}
    
    def get_descendant_leaves(node):
        if is_leaf[node]:
            if node in relevant_leaves:
                descendant_leaves[node].add(node)
            return descendant_leaves[node]
        
        for child in children_map.get(node, []):
            descendant_leaves[node].update(get_descendant_leaves(child))
        return descendant_leaves[node]

    for r in roots:
        get_descendant_leaves(r)

    leaf_labels = {row['node_id']: row['type'] for _, row in df.iterrows() if is_leaf[row['node_id']]}

    # 4. Helper function for Schwartz et al. subset clumpiness at node v
    def evaluate_node_clumpiness(root_node, target_labels):
        target_set = set(target_labels)
        n = len(target_set)
        
        sub_leaves = [l for l in descendant_leaves[root_node] if l in relevant_leaves]
        total_T = len(sub_leaves)
        if total_T == 0:
            return np.nan
            
        label_leaf_counts = {lbl: 0 for lbl in target_set}
        for l in sub_leaves:
            for lbl in leaf_labels.get(l, []):
                if lbl in label_leaf_counts:
                    label_leaf_counts[lbl] += 1
                    
        sub_inners = []
        def gather_inners(n_id):
            # Only gather actual internal nodes within this subtree
            if not is_leaf[n_id]:
                sub_inners.append(n_id)
                for c in children_map.get(n_id, []):
                    gather_inners(c)
        gather_inners(root_node)
        
        if not sub_inners:
            return np.nan
        
        total_I = len(sub_inners)
        c_score = 0
        
        for lbl in target_set:
            total_Li = label_leaf_counts[lbl]
            y_i = total_Li / total_T
            if y_i == 0:
                continue
                
            x_sum = 0
            for v in sub_inners:
                has_label = any(lbl in leaf_labels.get(l, []) for l in descendant_leaves[v] if l in sub_leaves)
                delta_v = 1 if has_label else 0
                
                if delta_v == 1:
                    w_v = 0
                    for l in descendant_leaves[v]:
                        if l in sub_leaves and lbl in leaf_labels.get(l, []):
                            path_weight = 1.0
                            curr = l
                            path_nodes = []
                            while curr != v and curr in parent_map:
                                curr = parent_map[curr]
                                if curr != l:
                                    path_nodes.append(curr)
                            
                            for j in path_nodes:
                                c_j = num_children.get(j, 1)
                                if c_j > 0:
                                    path_weight *= (1.0 / c_j)
                            w_v += path_weight
                    x_sum += (delta_v * w_v)
                    
            x_i = x_sum / total_I
            c_score += (x_i / y_i) if y_i > 0 else 0
            
        return (1.0 / n) * (c_score ** (1.0 / n))

    # 5. Assign columns to dataframe (default to NaN)
    for combo in label_combos:
        col_name = f"clump_{combo[0]}_{combo[1]}"
        df[col_name] = np.nan

    # Iterate through all nodes, but strictly skip leaves
    for node, leaf in is_leaf.items():
        if leaf:
            continue  # Skip leaves completely so they remain NaN
            
        matching_indices = df.index[df['node_id'] == node].tolist()
        if not matching_indices:
            continue
        idx = matching_indices[0]
        
        for combo in label_combos:
            col_name = f"clump_{combo[0]}_{combo[1]}"
            score = evaluate_node_clumpiness(node, combo)
            df.at[idx, col_name] = score

    return df

In [15]:
calculate_exact_node_clumpiness(swc_df=pd.read_csv(os.path.join("data","input_swc","simplified","720575940596125868.csv"), index_col=0))

,node_id,swc_type,x,y,z,r,parent,type,clump_post_post,clump_post_pre,clump_pre_pre
0,1,1,724748.00,275601.75,242456.56,1145,-1,[],0.126198,0.250178,0.124158
1,2,6,697556.00,266771.84,206129.83,20,62,[],NaN,NaN,NaN
2,11,6,698318.25,267801.38,204237.75,0,81,[],NaN,NaN,NaN
3,27,6,698629.10,264419.62,206938.75,88,62,[post],NaN,NaN,NaN
4,62,5,699790.44,264352.70,205602.31,297,81,[],0.500000,0.353553,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
202,1341,5,712254.44,267158.94,223100.83,499,1337,[pre],0.229004,0.322528,0.187093
203,1346,6,712464.30,271348.60,221820.14,102,1314,[],NaN,NaN,NaN
204,1385,6,712884.44,273480.22,221955.62,0,1281,[],NaN,NaN,NaN
205,1453,6,714310.70,265552.94,217135.05,91,1454,[],NaN,NaN,NaN
